In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/processed/AZ-628_expression_response.csv")

non_gene_cols = [
    'IDs', 'Drug.Name', 'ModelID', 'Response',
    'screen', 'dose', 'repurposing_target', 'MOA', 'Synonyms'
]

X = df.drop(columns=[col for col in non_gene_cols if col in df.columns])
X = X.select_dtypes(include='number')
y = df['Response']


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42
)


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ('feature_selection', SelectKBest(score_func=f_regression, k=2000)),
    ('scaler', StandardScaler())
])


In [5]:
X_train_processed = pipeline.fit_transform(X_train, y_train)
X_val_processed = pipeline.transform(X_val)


In [6]:
from sklearn.linear_model import Ridge

baseline_model = Ridge()
baseline_model.fit(X_train_processed, y_train)

y_pred = baseline_model.predict(X_val_processed)

In [7]:
from sklearn.metrics import mean_squared_error, r2_score

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
r2 = r2_score(y_val, y_pred)

print(f"RMSE: {rmse:.3f}")
print(f"R^2: {r2:.3f}")

RMSE: 1.078
R^2: 0.324


In [8]:
import joblib
joblib.dump(pipeline, "../models/feature_pipeline.pkl")
print("Pipeline saved successfully.")


Pipeline saved successfully.
